# Silver — Limpieza y enriquecimiento
**Pipeline stage:** `SILVER`

Silver aplica las reglas de calidad definidas en el Data Contract:
- Estrategias de nulos por columna (documentadas).
- Filtros de rangos físicos.
- Columnas derivadas: `spectral_class`, `hr_region`, `hab_zone_flag`.

In [1]:
import pandas as pd, numpy as np
import duckdb, pyarrow.parquet as pq, pyarrow as pa
import os

BRONZE_PATH = "../data/bronze/bronze_stellar.parquet"
df = pd.read_parquet(BRONZE_PATH)
print(f"Bronze cargado: {df.shape}")
df.head(2)

Bronze cargado: (6153, 18)


,pl_name,hostname,n_stars,n_planets,dist_pc,st_spectype,st_teff_k,st_rad_rsun,st_mass_msun,st_lum_log,st_logg,st_met_dex,pl_orbper_d,pl_rad_rearth,pl_mass_mearth,pl_eqt_k,disc_year,disc_method
0,Kepler-1167 b,Kepler-1167,1,1,820.905,None,4971.0,0.750,0.790,-0.53589,4.600,-0.05,1.003934,1.710000,3.57,1419.0,2016,Transit
1,Kepler-1740 b,Kepler-1740,1,1,1061.770,None,5705.0,0.905,0.943,-0.07942,4.499,-0.06,8.172400,3.323214,11.00,858.0,2021,Transit


## Filtros de rangos físicos

In [2]:
# Rangos físicamente válidos (decisión documentada en decisions_log.md)
FILTERS = {
    "st_teff_k"    : (2000,  50000),   # K: enanas rojas hasta estrellas O calientes
    "st_rad_rsun"  : (0.05,  1500),    # R☉: enanas marrones hasta hipergigantes
    "st_mass_msun" : (0.05,  300),     # M☉: límite fusión hasta estrella más masiva conocida
    "pl_orbper_d"  : (0.1,   1e6),     # días: un período mínimo real
    "pl_rad_rearth": (0.3,   30),      # R⊕: menor que Mercurio hasta ~3× Júpiter
}

n_before = len(df)
for col, (lo, hi) in FILTERS.items():
    mask = df[col].isnull() | df[col].between(lo, hi)
    removed = (~mask).sum()
    df = df[mask].copy()
    if removed > 0:
        print(f"Removidas {removed:4d} filas por {col} fuera de [{lo}, {hi}]")

print(f"\nFilas antes : {n_before:,}")
print(f"Filas después: {len(df):,}  (removidas: {n_before - len(df):,})")

Removidas    5 filas por st_teff_k fuera de [2000, 50000]
Removidas    4 filas por st_rad_rsun fuera de [0.05, 1500]
Removidas   12 filas por st_mass_msun fuera de [0.05, 300]
Removidas    4 filas por pl_orbper_d fuera de [0.1, 1000000.0]
Removidas    5 filas por pl_rad_rearth fuera de [0.3, 30]

Filas antes : 6,153
Filas después: 6,123  (removidas: 30)


## Estrategias de nulos

In [3]:
# Estrellas sin tipo espectral -> marcado como 'Unknown'
df["st_spectype"] = df["st_spectype"].fillna("Unknown")

# Columnas con alta completitud: rellenar con mediana por metodo descubrimiento
for col in ["st_teff_k", "st_mass_msun", "st_rad_rsun", "st_lum_log"]:
    mediana = df.groupby("disc_method")[col].transform("median")
    filled  = df[col].isna().sum()
    df[col] = df[col].fillna(mediana)
    print(f"  {col}: {filled} nulos rellenados con mediana por disc_method")

# pl_eqt_k: estimación física T_eq = T_eff * sqrt(R* / 2a) — aprox con albedo 0.3
# Si hay nulos se dejan (fact puede tener nulos en cols)
print(f"\nNulos restantes en columnas clave:")
key_cols = ["hostname","pl_name","st_teff_k","st_mass_msun","st_rad_rsun","st_lum_log"]
print(df[key_cols].isnull().sum().to_string())

  st_teff_k: 280 nulos rellenados con mediana por disc_method
  st_mass_msun: 8 nulos rellenados con mediana por disc_method
  st_rad_rsun: 298 nulos rellenados con mediana por disc_method
  st_lum_log: 294 nulos rellenados con mediana por disc_method

Nulos restantes en columnas clave:
hostname        0
pl_name         0
st_teff_k       7
st_mass_msun    0
st_rad_rsun     7
st_lum_log      7


## Columnas derivadas (enriquecimiento científico)

In [4]:
# Clase espectral principal (primera letra del tipo espectral)
def extract_spectral_class(s):
    if pd.isna(s) or s in ("Unknown", "nan", ""):
        return "Unknown"
    s = str(s).strip()
    if s[0] in "OBAFGKML":
        return s[0]
    return "Unknown"

df["spectral_class"] = df["st_spectype"].apply(extract_spectral_class)
print("Distribución de clases espectrales:")
print(df["spectral_class"].value_counts().to_string())

Distribución de clases espectrales:
spectral_class
Unknown    3836
G           780
K           646
M           542
F           286
A            24
B             8
L             1


In [5]:
# Región del Diagrama HR (usando Teff y luminosidad)
def hr_region(row):
    teff = row["st_teff_k"]
    lum  = row["st_lum_log"]      # log(L/L☉)
    logg = row["st_logg"]
    if pd.isna(teff) or pd.isna(lum):
        return "Unknown"
    if logg > 4.0 and teff < 6000:
        return "Main_Sequence_Cool"
    elif logg > 4.0 and teff >= 6000:
        return "Main_Sequence_Hot"
    elif 3.5 < logg <= 4.0:
        return "Subgiant"
    elif 2.5 < logg <= 3.5:
        return "Giant"
    elif logg <= 2.5:
        return "Supergiant"
    return "Unknown"

df["hr_region"] = df.apply(hr_region, axis=1)
print("Distribución de regiones HR:")
print(df["hr_region"].value_counts().to_string())

Distribución de regiones HR:
hr_region
Main_Sequence_Cool    4354
Main_Sequence_Hot     1036
Unknown                303
Subgiant               195
Giant                  160
Supergiant              75


In [6]:
# Flag de zona habitable (simplificado: T_eq entre 200-320 K)
df["hab_zone_flag"] = df["pl_eqt_k"].between(200, 320).astype("Int64")
n_hz = df["hab_zone_flag"].sum()
print(f"Planetas en zona habitable (estimado): {n_hz}")

# 3d. Categoría de tamaño planetario (clasificación Fulton)
def planet_size_class(r):
    if pd.isna(r): return "Unknown"
    if r < 1.5:    return "Rocky"
    elif r < 2.5:  return "Super-Earth"
    elif r < 4.0:  return "Sub-Neptune"
    elif r < 6.0:  return "Neptune-like"
    else:          return "Giant"

df["pl_size_class"] = df["pl_rad_rearth"].apply(planet_size_class)
print("\nClasificación de tamaño planetario:")
print(df["pl_size_class"].value_counts().to_string())

Planetas en zona habitable (estimado): 162

Clasificación de tamaño planetario:
pl_size_class
Giant           2034
Super-Earth     1561
Sub-Neptune     1193
Rocky            991
Neptune-like     295
Unknown           49


## Guardar Silver

In [7]:
os.makedirs("../data/silver", exist_ok=True)
SILVER_PATH = "../data/silver/silver_planet.parquet"
SILVER_CSV  = "../data/silver/silver_planet.csv"

table = pa.Table.from_pandas(df, preserve_index=False)
import pyarrow.parquet as pq
pq.write_table(table, SILVER_PATH, compression="snappy")
df.to_csv(SILVER_CSV, index=False)

print(f"Silver Parquet: {SILVER_PATH}  ({os.path.getsize(SILVER_PATH)/1024:.0f} KB)")
print(f"Silver CSV   : {SILVER_CSV}  ({os.path.getsize(SILVER_CSV)/1024:.0f} KB)")
print(f"Filas        : {len(df):,}")
print(f"Columnas     : {df.shape[1]}")
print(f"Nuevas cols  : spectral_class, hr_region, hab_zone_flag, pl_size_class")

Silver Parquet: ../data/silver/silver_planet.parquet  (385 KB)
Silver CSV   : ../data/silver/silver_planet.csv  (929 KB)
Filas        : 6,123
Columnas     : 22
Nuevas cols  : spectral_class, hr_region, hab_zone_flag, pl_size_class


## Reporte de calidad (evidencia)

In [8]:
con = duckdb.connect()

report = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT hostname) AS unique_stars,
    COUNT(DISTINCT pl_name) AS unique_planets,
    ROUND(AVG(st_teff_k), 1) AS avg_teff_k,
    ROUND(AVG(st_mass_msun), 3) AS avg_mass_msun,
    ROUND(AVG(st_lum_log), 3) AS avg_lum_log,
    SUM(CASE WHEN hab_zone_flag=1 THEN 1 END) AS hz_planets,
    COUNT(CASE WHEN spectral_class='Unknown' THEN 1 END) AS unknown_spectype
FROM read_parquet('{SILVER_PATH}')
""").df()

print("=== Reporte de calidad SILVER ===")
print(report.T.to_string())

=== Reporte de calidad SILVER ===
                         0
total_rows        6123.000
unique_stars      4557.000
unique_planets    6123.000
avg_teff_k        5385.400
avg_mass_msun        0.939
avg_lum_log         -0.118
hz_planets         162.000
unknown_spectype  3836.000


## Checkpoint SILVER
| Check | Estado |
|---|---|
| Filtros físicos aplicados | ✓ |
| Nulos tratados (estrategia documentada) | ✓ |
| `spectral_class` derivada | ✓ |
| `hr_region` derivada | ✓ |
| `hab_zone_flag` calculado | ✓ |
| Silver guardado (Parquet + CSV) | ✓ |